In [ ]:
import requests
import json

from dataclasses import dataclass
from datetime import datetime

from time import sleep

import sys
import os
from urllib.parse import urlsplit, urlunsplit, urljoin

# Get the current directory of the notebook
notebook_dir = os.getcwd()

# Get the parent directory (assuming the notebook is in folder B)
parent_dir = os.path.abspath(os.path.join(notebook_dir, os.pardir))

# Add the parent directory to sys.path
sys.path.append(parent_dir)

from monitoring import strategies as strats
from monitoring.strategies import URL, _get_content_with_css_selector

In [ ]:
# ?threadmark_category_id=1&min=1&max=46
# https://forums.spacebattles.com/threads/skitterdoc-2077-worm-cyberpunk-2077-crossover.1052653/threadmarks-load-range?threadmark_category_id=1&min=1&max=46
test_urls = [
    'https://forums.spacebattles.com/threads/skitterdoc-2077-worm-cyberpunk-2077-crossover.1052653/threadmarks',
    'https://forums.spacebattles.com/threads/skitterdoc-2077-worm-cyberpunk-2077-crossover.1052653/',
    'https://forums.sufficientvelocity.com/threads/skitterdoc-2077.109765/',
]

In [ ]:
responses: list[requests.Response] = []
for url in test_urls:
    parsed = urlsplit(url)
    req_url = parsed.scheme + "://" + parsed.netloc + '/' + parsed.path.replace('threadmarks', '')
    req_url += '/threadmarks-load-range'
    req_url += '?threadmark_category_id=1'

    reponse = requests.get(req_url)
    responses.append(reponse)

    print(req_url)
    print(reponse)

    sleep(0.1)
    

In [ ]:
"Threadmarks" in reponse.text and "Tinker, Taylor, Entrepreneur, Spy" in reponse.text

In [ ]:
print(reponse.text[:200])

In [ ]:
print(f"{test_urls[0] =  }")
print("target =         https://forums.sufficientvelocity.com/threads/skitterdoc-2077.109765/threadmarks-load-range?threadmark_category_id=1")

In [ ]:
# document.querySelector('.structItem').innerText
# "If she was the butterfly then am I just a moth?
	
# Words
# 5.6k
# 	Nov 11, 2022"
# document.querySelector('.structItem a')
# <a class="" href="/threads/skitterdoc-2077….1052653/#post-88276522" data-tp-primary="on" data-xf-init="preview-tooltip" data-preview-url="/posts/88276522/preview-threadmark">

# document.querySelector('.structItem a').innerText
# "If she was the butterfly then am I just a moth?"
# document.querySelector('.structItem time').innerText
# "Nov 11, 2022"
# document.querySelector('.structItem dd').innerText 

In [ ]:
@dataclass
class SThreadmarkInfo:
    title: str | None = None
    word_count: str | None = None
    pub_date: datetime | None = None
    link: URL | None = None

    def to_json(self) -> str:
        json_dict = {
            'title': self.title,
            'word_count': self.word_count,
            'pub_date': None if self.pub_date is None else self.pub_date.isoformat(),
            'link': self.link
        }
        return json.dumps(json_dict)

    @classmethod
    def from_json(cls, json_str: str) -> 'SThreadmarkInfo':
        json_dict = json.loads(json_str)
        pub_date = None if json_dict['pub_date'] is None else datetime.fromisoformat(json_dict['pub_date'])
        return cls(
            title=json_dict['title'],
            word_count=json_dict['word_count'],
            pub_date=pub_date,
            link=json_dict['link']
        )

    def __str__(self):
        return f"SThreadmark Information:\n" \
               f"- Title: {self.title}\n" \
               f"- Word Count: {self.word_count}\n" \
               f"- Pub Date: {self.pub_date}\n" \
               f"- Link: {self.link}"

marks: list[SThreadmarkInfo] = []
for response in responses:
    mark_tags = _get_content_with_css_selector(response.text, '.structItem--threadmark')

    base_url = response.url
    parsed_url = urlsplit(base_url)
    scheme = parsed_url.scheme
    netloc = parsed_url.netloc
    base_url = f"{scheme}://{netloc}"

    for mark_tag in mark_tags:
        # The title of the threadmark and the href/link are on the same HTML tag
        title = mark_tag.select_one('a')
        link = None
        if title is not None:
            link = title.attrs.get('href')
            title = title.text
            if link is not None:
                link = base_url + link

        pub_date = mark_tag.select_one('time')
        if pub_date is not None:
            if (pub_date := pub_date.attrs.get('datetime')) is not None:
                pub_date = datetime.strptime(pub_date, "%Y-%m-%dT%H:%M:%S%z")

        wordcount = mark_tag.select_one('dd')
        if wordcount is not None:
            wordcount = wordcount.text  
        
        mark_info = SThreadmarkInfo(
                title=title, 
                word_count=wordcount,
                pub_date=pub_date,
                link=link
            )

        if mark_info.link is not None:
            marks.append(mark_info)
    
    print(response.url, len(mark_tags))


In [ ]:
print(responses[0].url)
print(marks[-2:][0])

In [ ]:
marks[-1]